# Лабораторная работа №1. Метрики (пункт 5-6)

Вход: очищенный корпус `body_clean` из `1_eda.ipynb` (`artifacts/train_clean.csv`, `artifacts/test_clean.csv`) — тексты уже содержат технические токены `<s>`, `</s>`, `[url]`, `[num]`, `[hex]`, `[code]`, `[user]`, `[issueref]`, `[nonlatin]` вместо соответствующего шума.


In [1]:
import re
import time
from collections import Counter, defaultdict

import pandas as pd
from nltk import pos_tag
from nltk.corpus import wordnet as wn
from nltk.stem import PorterStemmer, WordNetLemmatizer

ARTIFACTS_DIR = "artifacts"

train = pd.read_csv(f"{ARTIFACTS_DIR}/train_clean.csv")
test = pd.read_csv(f"{ARTIFACTS_DIR}/test_clean.csv")
print(f"train: {len(train):,} документов, test: {len(test):,} документов")

WORD_RE = re.compile(r"[a-z]+")
# эти токены не имеет смысла подавать в стемминг/лемматизацию/поиск опечаток
TECH_TOKENS = {"s", "num", "url", "code", "user", "issueref", "hex", "nonlatin"}

def extract_words(text):
    if not isinstance(text, str):
        return []
    return [w for w in WORD_RE.findall(text) if w not in TECH_TOKENS and len(w) > 1]

train_tokens = []
for text in train["body_clean"]:
    train_tokens.extend(extract_words(text))

print(f"Токенов (без технических/структурных) в train: {len(train_tokens):,}")
raw_vocab = set(train_tokens)
print(f"Размер исходного словаря (без нормализации): {len(raw_vocab):,}")


train: 40,000 документов, test: 10,000 документов
Токенов (без технических/структурных) в train: 2,242,154
Размер исходного словаря (без нормализации): 86,644


## Шаг 5. Сравнительная нормализация: стемминг vs лемматизация

In [2]:
t0 = time.time()
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(w) for w in train_tokens]
stem_vocab = set(stemmed_tokens)
print(f"Стемминг: {time.time() - t0:.1f} сек, словарь = {len(stem_vocab):,} уникальных стемов")


Стемминг: 15.0 сек, словарь = 73,478 уникальных стемов


In [3]:
def wordnet_pos(tag):
    if tag.startswith("J"):
        return wn.ADJ
    if tag.startswith("V"):
        return wn.VERB
    if tag.startswith("R"):
        return wn.ADV
    return wn.NOUN  # по умолчанию существительное

t0 = time.time()
tagged_tokens = pos_tag(train_tokens)
print(f"POS-теггинг: {time.time() - t0:.1f} сек")

t0 = time.time()
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(w, wordnet_pos(tag)) for w, tag in tagged_tokens]
lemma_vocab = set(lemmatized_tokens)
print(f"Лемматизация: {time.time() - t0:.1f} сек, словарь = {len(lemma_vocab):,} уникальных лемм")


POS-теггинг: 42.7 сек
Лемматизация: 5.7 сек, словарь = 82,124 уникальных лемм


In [4]:
comparison = pd.DataFrame([
    {"Способ": "без нормализации", "Размер словаря": len(raw_vocab), "Сокращение, %": 0.0},
    {"Способ": "стемминг (Porter)", "Размер словаря": len(stem_vocab),
     "Сокращение, %": round((1 - len(stem_vocab) / len(raw_vocab)) * 100, 1)},
    {"Способ": "лемматизация (WordNet + POS)", "Размер словаря": len(lemma_vocab),
     "Сокращение, %": round((1 - len(lemma_vocab) / len(raw_vocab)) * 100, 1)},
])
comparison


,Способ,Размер словаря,"Сокращение, %"
0,без нормализации,86644,0.0
1,стемминг (Porter),73478,15.2
2,лемматизация (WordNet + POS),82124,5.2


Вывод 1: лемматизация даёт более умеренное и лингвистически корректное сокращение словаря (объединяет только реальные словоформы одной леммы), тогда как стемминг сокращает словарь сильнее, но за счёт грубого отбрасывания суффиксов — часть «сокращения» приходится не на объединение синонимичных форм, а на артефакты (over-stemming, склеивание разных по смыслу слов). Проверяется как численным сравнением размеров словарей выше, так и качественным разбором коллизий ниже (H6).

In [5]:
stem_to_words = defaultdict(set)
for original, stem in zip(train_tokens, stemmed_tokens):
    stem_to_words[stem].add(original)

collisions = [(stem, words) for stem, words in stem_to_words.items() if len(words) > 1]
collisions.sort(key=lambda x: -len(x[1]))

print(f"Стемов, к которым свелось больше одного исходного слова: {len(collisions):,} из {len(stem_vocab):,}")
print()
print("Топ-10 самых «нагруженных» стемов (потенциальный over-stemming):")
for stem, words in collisions[:10]:
    print(f"  '{stem}' <- {sorted(words)[:12]}{' ...' if len(words) > 12 else ''}  (всего форм: {len(words)})")


Стемов, к которым свелось больше одного исходного слова: 7,566 из 73,478

Топ-10 самых «нагруженных» стемов (потенциальный over-stemming):
  'gener' <- ['general', 'generality', 'generalizations', 'generalized', 'generalizes', 'generally', 'generals', 'generate', 'generated', 'generates', 'generating', 'generation'] ...  (всего форм: 20)
  'initi' <- ['initial', 'initialization', 'initializations', 'initialize', 'initialized', 'initializer', 'initializers', 'initializes', 'initializing', 'initially', 'initials', 'initiate'] ...  (всего форм: 20)
  'depend' <- ['depend', 'dependable', 'dependancies', 'dependancy', 'dependant', 'depende', 'depended', 'dependence', 'dependences', 'dependencies', 'dependency', 'dependent'] ...  (всего форм: 16)
  'activ' <- ['activ', 'activate', 'activated', 'activates', 'activating', 'activation', 'activations', 'activator', 'active', 'actived', 'actively', 'activism'] ...  (всего форм: 15)
  'author' <- ['author', 'authored', 'authoring', 'authorities', 

In [7]:
lemma_to_words = defaultdict(set)
for original, lemma in zip(train_tokens, lemmatized_tokens):
    lemma_to_words[lemma].add(original)

lemma_collisions = [(lemma, words) for lemma, words in lemma_to_words.items() if len(words) > 1]
lemma_collisions.sort(key=lambda x: -len(x[1]))

print(f"Лемм, к которым свелось больше одного исходного слова: {len(lemma_collisions):,} из {len(lemma_vocab):,}")
print()
print("Топ-10 самых «нагруженных» лемм (потенциальные коллизии лемматизации):")
for lemma, words in lemma_collisions[:10]:
    print(f"  '{lemma}' <- {sorted(words)[:12]}{' ...' if len(words) > 12 else ''}  (всего форм: {len(words)})")

Лемм, к которым свелось больше одного исходного слова: 4,360 из 82,124

Топ-10 самых «нагруженных» лемм (потенциальные коллизии лемматизации):
  'be' <- ['am', 'are', 'be', 'beed', 'been', 'being', 'is', 'was', 'were']  (всего форм: 9)
  'plan' <- ['plan', 'planed', 'planes', 'planing', 'planned', 'planning', 'plans']  (всего форм: 7)
  'occur' <- ['occur', 'occured', 'occures', 'occuring', 'occurred', 'occurring', 'occurs']  (всего форм: 7)
  'refer' <- ['refer', 'refered', 'referes', 'refering', 'referred', 'referring', 'refers']  (всего форм: 7)
  'run' <- ['ran', 'run', 'runed', 'runing', 'running', 'runs']  (всего форм: 6)
  'do' <- ['did', 'do', 'does', 'doing', 'done', 'dos']  (всего форм: 6)
  'have' <- ['had', 'has', 'have', 'haveing', 'haves', 'having']  (всего форм: 6)
  'log' <- ['log', 'loged', 'logged', 'logging', 'loging', 'logs']  (всего форм: 6)
  'throw' <- ['threw', 'throw', 'throwed', 'throwing', 'thrown', 'throws']  (всего форм: 6)
  'try' <- ['tried', 'tries', 'tr

Вывод 2: среди самых «нагруженных» стемов часто встречаются группы, где объединены и правомерные словоформы (`request`/`requests`/`requesting` → `request`), и слова, которые не являются формами друг друга, а просто случайно совпали после отбрасывания суффикса (классический пример over-stemming для алгоритма Porter — слова с разной семантикой сводятся к одному короткому стему). Лемматизатор такой проблемы почти не создаёт, так как опирается на словарь WordNet и часть речи, а не на отбрасывание суффиксов по правилам.

**Вывод по применимости:** для технического IT-текста (GitHub Issues) с обилием сокращений, английских глагольных форм и специфической терминологии лемматизация выглядит предпочтительнее — она даёт контролируемое сокращение словаря без риска объединить семантически разные термины. Стемминг может использоваться как быстрая грубая нормализация (например, для полнотекстового поиска), но для семантического анализа его агрессивность — риск.

## Шаг 6. Поиск опечаток через собственную реализацию расстояния Левенштейна

In [8]:
def levenshtein(a: str, b: str) -> int:
    '''Расстояние Левенштейна с DP на двух строках (O(min(len(a), len(b))) памяти
    вместо полной матрицы O(len(a) * len(b))).'''
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    previous_row = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current_row = [i] + [0] * len(b)
        for j, char_b in enumerate(b, start=1):
            if char_a == char_b:
                current_row[j] = previous_row[j - 1]
            else:
                current_row[j] = 1 + min(
                    previous_row[j - 1],  # замена
                    previous_row[j],      # удаление
                    current_row[j - 1],   # вставка
                )
        previous_row = current_row
    return previous_row[-1]


**Оптимизация по памяти:** классическая реализация хранит полную матрицу `(len(a)+1) x (len(b)+1)`, то есть `O(n*m)` памяти. Здесь используются только две строки матрицы (`previous_row` и `current_row`), поэтому память — `O(min(n, m))` (короче из двух строк всегда выбирается как измерение внутреннего цикла).

**Проблема прямого перебора по словарю:** полный перебор всех пар токенов словаря — `O(V^2)` сравнений, что для `V` в десятки тысяч (см. п.5) даёт до ~10^9 пар — нереалистично для одного ноутбука. Кроме того, короткие слова (3-4 буквы) почти всегда оказываются на расстоянии 1-2 от множества случайных других коротких слов (например `the`/`to`), что даёт огромное количество бессмысленных «совпадений», а не настоящих опечаток/словоформ.

Поэтому поиск строится так:
1. **Блокировка кандидатов** — сравниваются только слова с одинаковой первой буквой и близкой длиной (разница ≤2), это резко уменьшает число пар без потери настоящих опечаток (при расстоянии Левенштейна ≤2 длины строк не могут отличаться больше, чем на 2).
2. **Фильтр по длине слова (≥6 символов)** — отсекает короткие слова, для которых расстояние 1-2 практически не информативно.
3. **Фильтр по частоте (≥8 упоминаний в train)** — интересны опечатки/словоформы у слов, которые встречаются не единично, иначе результат статистически ненадёжен.

In [9]:
MIN_WORD_LEN = 6
MIN_COUNT = 8

vocab_counter = Counter(w for w in train_tokens)
typo_vocab = [w for w, c in vocab_counter.items() if c >= MIN_COUNT and len(w) >= MIN_WORD_LEN]
print(f"Словарь для поиска опечаток (длина >= {MIN_WORD_LEN}, частота >= {MIN_COUNT}): {len(typo_vocab):,} слов")

by_bucket = defaultdict(list)
for w in typo_vocab:
    by_bucket[(w[0], len(w))].append(w)

t0 = time.time()
candidate_pairs = 0
typo_pairs = []
for w in typo_vocab:
    L = len(w)
    first_char = w[0]
    for length2 in (L - 2, L - 1, L, L + 1, L + 2):
        for w2 in by_bucket.get((first_char, length2), []):
            if w2 <= w:
                continue
            candidate_pairs += 1
            dist = levenshtein(w, w2)
            if dist in (1, 2):
                typo_pairs.append((w, w2, dist, vocab_counter[w] + vocab_counter[w2]))

print(f"Сравнено пар-кандидатов: {candidate_pairs:,} за {time.time() - t0:.1f} сек")
print(f"Найдено пар с расстоянием 1 или 2: {len(typo_pairs):,}")


Словарь для поиска опечаток (длина >= 6, частота >= 8): 7,929 слов
Сравнено пар-кандидатов: 1,214,220 за 12.4 сек
Найдено пар с расстоянием 1 или 2: 7,535


In [10]:
typo_pairs.sort(key=lambda x: -x[3])
top10 = pd.DataFrame(typo_pairs[:30], columns=["слово 1", "слово 2", "расстояние Левенштейна", "суммарная частота"])
top10


,слово 1,слово 2,расстояние Левенштейна,суммарная частота
0,version,versions,1,9923
1,version,vision,2,8995
2,version,versioned,2,8987
3,verison,version,2,8980
4,should,shouldn,1,7912
5,should,showed,2,7733
6,shield,should,2,7695
7,create,created,1,4341
8,current,currently,2,4314
9,module,modules,1,3776


Вывод: в корпусе действительно существуют пары слов с расстоянием Левенштейна 1-2, соответствующие либо настоящим опечаткам (например, варианты написания технических терминов), либо словоформам (единственное/множественное число, отглагольные формы).

## Общие выводы по ноутбуку

1. Лемматизация даёт более контролируемое и лингвистически обоснованное сокращение словаря, чем стемминг; стемминг сокращает словарь сильнее, но ценой получения коллизий между несвязанными словами.
2. Прямой `O(V^2)` перебор по словарю нерационален — использована блокировка кандидатов по первой букве и длине, что делает поиск опечаток практически осуществимым и одновременно отфильтровывает мусорные совпадения между случайными короткими словами.